In [0]:
dbfs:/FileStore/shared_uploads/revathy.s@diggibyte.com/tmp2/sample_data_one_translated.xlsx
dbfs:/FileStore/shared_uploads/revathy.s@diggibyte.com/tmp2/Sample_data_one_1.xlsx
dbfs:/FileStore/shared_uploads/revathy.s@diggibyte.com/tmp2/Sample_data_one.xlsx

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import pandas as pd

In [0]:
import os
print(os.path.exists("/dbfs/FileStore/shared_uploads/revathy.s@diggibyte.com/tmp2/sample_data_one_translated.xlsx"))

In [0]:
%pip install openpyxl

In [0]:
dbutils.library.restartPython()

In [0]:
file_path = "/dbfs/FileStore/shared_uploads/revathy.s@diggibyte.com/tmp/sample_data_one_translated-1.xlsx"

df = pd.read_excel(file_path, header=[0, 1]) 

# Flatten column headers 
df.columns = ['_'.join(map(str, col)).strip('_') for col in df.columns]

df.rename(columns={df.columns[0]: "Model"}, inplace=True)

df = df.dropna()

for col in df.columns:
    if col != "Model":
        df[col] = pd.to_numeric(df[col], errors='coerce')

display(df)

In [0]:


# Define the schema
schema = StructType([
    StructField("Model", StringType(), True),  # Product model
    StructField("Taichung", IntegerType(), True),  # Shinko Mitsukoshi Taichung
    StructField("North_Station", IntegerType(), True),  # Shinko Mitsukoshi North Station
    StructField("Peach_Station", IntegerType(), True),  # Shinko Mitsukoshi Peach Station
    StructField("Chiayi", IntegerType(), True),  # Shinko Mitsukoshi Chiayi
    StructField("Simon", IntegerType(), True),  # Shinko Mitsukoshi Simon
    StructField("Left_Battalion", IntegerType(), True),  # Shinko Mitsukoshi Left Battalion
    StructField("A8", IntegerType(), True),  # Shinko Mitsukoshi A8
    StructField("Heavenly_Mother", IntegerType(), True),  # Shinko Mitsukoshi Heavenly Mother
    StructField("Revival", IntegerType(), True),  # SOGO Revival
    StructField("Hsinchu", IntegerType(), True),  # SOGO Hsinchu
    StructField("Three_Creations", IntegerType(), True),  # Taipei Three creations
    StructField("Tainan", IntegerType(), True),  # Outlet Tainan
    StructField("Kaohsiung", IntegerType(), True),  # Outlet Kaohsiung
    StructField("Total_Turnover", FloatType(), True),  # Total turnover
    StructField("Unit_Price", StringType(), True),  # Unit price (currency format)
    StructField("Total_Amount", StringType(), True),  # The total amount (currency format)
    StructField("Inventory_End_Of_Month", IntegerType(), True),  # Inventory at the end of the month
])

print(schema)

In [0]:
import pandas as pd

file_path = "/dbfs/FileStore/shared_uploads/revathy.s@diggibyte.com/tmp/sample_data_one_translated-1.xlsx"

# Load the Excel file
df_pandas = pd.read_excel(file_path, header=[0, 1]).apply(pd.to_numeric, errors='coerce')

# Convert Pandas dataframe to Spark dataframe
df= spark.createDataFrame(df_pandas)

# Show the data in Spark dataframe
display(df)

In [0]:
# Flatten multi-level headers (if applicable)
# Assuming column names need cleanup like replacing spaces or special characters
flattened_columns = [col_name.replace(" ", "_").replace("\n", "_") for col_name in df.columns]
df = df.toDF(*flattened_columns)

# Rename the first column to "Model"
first_column = df.columns[0]
df = df.withColumnRenamed(first_column, "Model")

# Drop completely blank rows (rows where all columns are null)
df = df.na.drop(how="all")

# Ensure numeric columns are coerced
for column_name in df.columns:
    if column_name != "Model":  # Skip non-numeric columns
        df = df.withColumn(
            column_name,
            when(col(column_name).cast("double").isNotNull(), col(column_name).cast("double")).otherwise(lit(None))
        )

# (Optional) Fill NaNs with 0 if desired
# df = df.fillna(0)

# Display the cleaned DataFrame
df.show()

In [0]:
import pandas as pd

# Path to the Excel file
file_path = "/dbfs/FileStore/shared_uploads/revathy.s@diggibyte.com/tmp2/sample_data_one_translated-2.xlsx"

# Read the Excel file with multi-level headers
df = pd.read_excel(file_path, header=[0, 1], engine="openpyxl")

# Create a function to flatten multi-level column names
def flatten_columns(columns):
    flattened = []
    for col in columns:
        # If the second-level header is "Unnamed", use only the top-level header
        if "Unnamed" in col[1]:
            flattened.append(col[0])
        else:
            # Otherwise, combine the top-level and second-level headers
            flattened.append(f"{col[0]}_{col[1]}")
    return flattened

# Flatten the columns
df.columns = flatten_columns(df.columns)

# Drop rows with all NaN values
df = df.dropna(how="all", axis=0)

# Drop meta rows like "Company-wide" if they exist
if df.columns[0] == "Model":
    df = df[~df["Model"].str.contains("Company-wide", na=False)]

# Save the cleaned DataFrame if needed
df.to_csv("flattened_excel_file.csv", index=False)

# Display the cleaned DataFrame for verification
print(df)

display(df)


In [0]:
import pandas as pd

# Path to the Excel file
file_path = "/dbfs/FileStore/shared_uploads/revathy.s@diggibyte.com/tmp2/sample_data_one_translated-2.xlsx"

# Read the Excel file with multi-level headers
df = pd.read_excel(file_path, header=[0, 1], engine="openpyxl")

# Flatten the columns by combining multi-level headers
df.columns = [
    f"{col[0]}_{col[1]}" if "Unnamed" not in col[1] else f"{col[0]}"
    for col in df.columns
]

# Fill any remaining "Unnamed" top-level headers with the second-level names
df.columns = [col.replace("Unnamed_", "").strip() for col in df.columns]

# Drop rows with all NaN values
df = df.dropna(how="all", axis=0)

# Display the flattened DataFrame
print(df)

# Save the cleaned file if needed
df.to_csv("flattened_excel_file.csv", index=False)

display(df)


In [0]:
file_path = "/dbfs/FileStore/shared_uploads/revathy.s@diggibyte.com/tmp2/sample_data_one_translated-2.xlsx"  # Replace with your file path
df = pd.read_excel(file_path, header=[0,1], engine="openpyxl")

df.columns = ['-'.join([str(i) for i in col if pd.notnull(i)]) for col in df.columns]; df = df.apply(pd.to_numeric, errors='coerce')
display(df)

In [0]:
import pandas as pd

# Load the Excel file with Pandas
file_path = "/dbfs/FileStore/shared_uploads/revathy.s@diggibyte.com/tmp/sample_data_one_translated-1.xlsx"  # Replace with your file path
df = pd.read_excel(file_path, header=[0,1], engine="openpyxl")  # Reading multi-level headers

# Flatten multi-level headers
df.columns = ['_'.join(map(str, col)).strip() for col in df.columns.values]; df = df.apply(pd.to_numeric, errors='coerce')

# df.rename(columns = lambda x:x.replace("Unnamed: 0_level_1", "Model"), inplace = True)

# Save the cleaned file for PySpark ingestion
cleaned_file_path = "/dbfs/tmp/cleaned_excel_file.csv"
df.to_csv(cleaned_file_path, index=False)

display(df)

In [0]:
import pandas as pd

# Load the Excel file with Pandas
file_path = "/dbfs/FileStore/shared_uploads/revathy.s@diggibyte.com/tmp/sample_data_one_translated-1.xlsx"  # Replace with your file path
df = pd.read_excel(file_path, header=[0,1])  # Reading multi-level headers

# Check if there are nulls in the "Model" column and inspect data
print("Before cleaning:")
print(df.head())
 
# Drop completely blank rows (if any)
df = df.dropna(how='all')
 
# Ensure "Model" column is read properly
df.rename(columns={"Unnamed: 0": "Model"}, inplace=True)  # Replace with the correct column header if needed
 
 # Fix numeric columns by coercing them to a consistent data type
for col in df.columns:
    if col != "Model":  # Skip non-numeric columns like "Model"
        df[col] = pd.to_numeric(df[col], errors='coerce')  # Converts non-numeric values to NaN
 
# Fill NaNs if needed or leave them as null
df = df.fillna(0)  # Replace NaN with 0 if desired
 
# Display the cleaned DataFrame in Databricks
display(df)

In [0]:
import pandas as pd

file_path = "/dbfs/FileStore/shared_uploads/revathy.s@diggibyte.com/tmp/sample_data_one_translated-1.xlsx"

# Load the Excel file
df = pd.read_excel(file_path, header=[0, 1])  # Multi-level headers

# Flatten column headers if multi-level
df.columns = ['_'.join(map(str, col)).strip('_') for col in df.columns]

# Inspect columns
print("Columns:", df.columns)

# Rename the first column to "Model"
df.rename(columns={df.columns[0]: "Model"}, inplace=True)

# Drop completely blank rows
df = df.dropna(how='all')

# Ensure numeric columns are coerced
for col in df.columns:
    if col != "Model":
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Fill NaNs if desired
df = df.fillna(0)

# Display the cleaned DataFrame
display(df)


In [0]:
# Read the cleaned CSV file with PySpark
cleaned_file_path = "dbfs:/tmp/cleaned_excel_file.csv"
df = spark.read.csv(cleaned_file_path, header=True, inferSchema=True)

In [0]:
from pyspark.sql.functions import col

# Dynamically add prefixes based on groups
def add_prefix(df, prefix_dict):
    for prefix, cols in prefix_dict.items():
        for col_name in cols:
            new_name = f"{prefix}_{col_name}"
            df = df.withColumnRenamed(col_name, new_name)
    return df

# Example usage: Automatically detect prefixes based on column names
columns = df.columns
prefix_dict = {
    "Shinko Mitsukoshi": [col for col in columns if "Shinko Mitsukoshi" in col],
    "SOGO": [col for col in columns if "SOGO" in col],
    "outlet": [col for col in columns if "outlet" in col],
}

# Apply the dynamic mapping
renamed_df = add_prefix(df, prefix_dict)
display(renamed_df)


In [0]:
%python
# Identify columns to unpivot
unpivot_columns = [col for col in df.columns if "Shinko_Mitsukoshi" in col or "SOGO" in col or "outlet" in col]

# Define the identifier columns (e.g., Model or other static columns)
id_columns = [col for col in df.columns if col not in unpivot_columns]

# Unpivot using selectExpr and stack
stack_expr = f"stack({len(unpivot_columns)}, " + ", ".join(
    [f"'{col}', `{col}`" for col in unpivot_columns]
) + ") as (Category, Value)"

unpivoted_df = df.selectExpr(*[f"`{col}`" for col in id_columns], stack_expr)

# Display the unpivoted DataFrame
display(unpivoted_df)
